# Notebook 01 — Data Download and Preprocessing

This notebook:
1. Clones `cobanov/shakespeare-dataset` for EDA raw texts
2. Downloads two HuggingFace parallel corpora
3. Merges, deduplicates, and splits the data
4. Builds bidirectional chat-format records and saves as JSONL

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
from datasets import load_dataset

# Add src/ to path so we can import project utilities
sys.path.insert(0, str(Path('..').resolve()))
from src.data_utils import make_record, save_jsonl, build_training_records

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT         = Path('..').resolve()
DATA_DIR     = ROOT / 'data'
RAW_DIR      = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
HF_CACHE     = RAW_DIR / 'huggingface_cache'
SHAK_REPO    = RAW_DIR / 'cobanov_shakespeare'

for d in [PROCESSED_DIR, HF_CACHE, SHAK_REPO]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths configured.')

Paths configured.


## 1. Clone cobanov/shakespeare-dataset (raw texts for EDA)

In [2]:
REPO_URL = 'https://github.com/cobanov/shakespeare-dataset.git'

if not (SHAK_REPO / '.git').exists():
    print('Cloning cobanov/shakespeare-dataset ...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(SHAK_REPO)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('git clone failed')
    print('Clone complete.')
else:
    print('Repository already cloned.')

raw_text_files = list(SHAK_REPO.rglob('*.txt'))
print(f'Found {len(raw_text_files)} .txt files in the Shakespeare repo.')

Repository already cloned.
Found 42 .txt files in the Shakespeare repo.


## 2. Load Parallel Corpora from HuggingFace

In [3]:
# ── ayaan04/english-to-shakespeare ─────────────────────────────────────────
print('Loading ayaan04/english-to-shakespeare ...')
ds_ayaan = load_dataset(
    'ayaan04/english-to-shakespeare',
    cache_dir=str(HF_CACHE)
)
df_ayaan = ds_ayaan['train'].to_pandas()

# The dataset uses 'input' / 'output' column names
df_ayaan = df_ayaan.rename(columns={'input': 'modern', 'output': 'shakespeare'})
df_ayaan = df_ayaan[['modern', 'shakespeare']].dropna().reset_index(drop=True)

print(f'ayaan04 dataset: {len(df_ayaan):,} rows')
df_ayaan.head(3)

Loading ayaan04/english-to-shakespeare ...


Repo card metadata block was not found. Setting CardData to empty.


ayaan04 dataset: 18,395 rows


,modern,shakespeare
0,I have half a mind to hit you before you speak...,I have a mind to strike thee ere thou speak'st .
1,"But if Antony is alive , healthy , friendly wi...","Yet if thou say Antony lives , is well , Or fr..."
2,"Madam , he's well .","Madam , he's well ."


In [4]:
# ── Roudranil conversational dataset ───────────────────────────────────────
print('Loading Roudranil/shakespearean-and-modern-english-conversational-dataset ...')
ds_roudranil = load_dataset(
    'Roudranil/shakespearean-and-modern-english-conversational-dataset',
    cache_dir=str(HF_CACHE)
)
print('Splits available:', list(ds_roudranil.keys()))

df_r_train = ds_roudranil['train'].to_pandas()
df_r_test  = ds_roudranil['test'].to_pandas()

# Rename to canonical column names
RENAME = {'translated_dialog': 'modern', 'og_response': 'shakespeare'}
df_r_train = df_r_train.rename(columns=RENAME)[['modern', 'shakespeare']].dropna().reset_index(drop=True)
df_r_test  = df_r_test.rename(columns=RENAME)[['modern', 'shakespeare']].dropna().reset_index(drop=True)

print(f'Roudranil train: {len(df_r_train):,} rows')
print(f'Roudranil test:  {len(df_r_test):,}  rows (held-out)')
df_r_train.head(3)

Loading Roudranil/shakespearean-and-modern-english-conversational-dataset ...
Splits available: ['train', 'test']
Roudranil train: 5,272 rows
Roudranil test:  3,515  rows (held-out)


,modern,shakespeare
0,You'd better be quiet.,Thou hast not half that power to do me harm...
1,"Oh, good sir, I am.","I pray thee, mark me. I, thus neglecting..."
2,Dear lady,"Give me leave, beseech you. I did send, ..."


## 3. Merge, Deduplicate, and Split

In [5]:
# Combine ayaan04 + Roudranil train
df_combined = pd.concat([df_ayaan, df_r_train], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['modern']).reset_index(drop=True)
print(f'Combined (deduped): {len(df_combined):,} rows')

# 90/10 train/val split
val_size  = int(0.10 * len(df_combined))
df_val    = df_combined.sample(n=val_size, random_state=42)
df_train  = df_combined.drop(df_val.index).reset_index(drop=True)
df_val    = df_val.reset_index(drop=True)

# Held-out test = Roudranil test split
df_test   = df_r_test.copy()

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')

Combined (deduped): 22,743 rows
Train: 20,469  Val: 2,274  Test: 3,515


In [6]:
# ── Length filtering ────────────────────────────────────────────────────────
MAX_CHARS = 512

def length_filter(df, max_chars=MAX_CHARS):
    mask = (
        (df['modern'].str.len() <= max_chars) &
        (df['shakespeare'].str.len() <= max_chars)
    )
    return df[mask].reset_index(drop=True)

df_train = length_filter(df_train)
df_val   = length_filter(df_val)
# Keep test set unfiltered to evaluate on the natural distribution

print(f'After length filter (≤{MAX_CHARS} chars):')
print(f'  Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')

After length filter (≤512 chars):
  Train: 20,042  Val: 2,234  Test: 3,515


## 4. Build Chat-Format Records and Save

In [7]:
# Training: bidirectional (mod→shak + shak→mod)
records_train = build_training_records(df_train, bidirectional=True)

# Val and test: mod→shak only (primary evaluation direction)
records_val  = build_training_records(df_val,  bidirectional=False)
records_test = build_training_records(df_test, bidirectional=False)

print(f'Train records (bidirectional): {len(records_train):,}')
print(f'Val records:   {len(records_val):,}')
print(f'Test records:  {len(records_test):,}')

# Preview one record
import pprint
pprint.pprint(records_train[0])

Train records (bidirectional): 40,084
Val records:   2,234
Test records:  3,515
{'messages': [{'content': 'You are an expert translator of Modern English into '
                          'Shakespearean English. Translate the following '
                          'modern English text into authentic Shakespearean '
                          'style, preserving the meaning while using '
                          'appropriate Early Modern English vocabulary, '
                          'grammar, and poetic diction.',
               'role': 'system'},
              {'content': 'I have half a mind to hit you before you speak '
                          'again .',
               'role': 'user'},
              {'content': "I have a mind to strike thee ere thou speak'st .",
               'role': 'assistant'}]}


In [8]:
# Save to disk
save_jsonl(records_train, PROCESSED_DIR / 'train.jsonl')
save_jsonl(records_val,   PROCESSED_DIR / 'val.jsonl')
save_jsonl(records_test,  PROCESSED_DIR / 'test.jsonl')
print('Saved processed splits to', PROCESSED_DIR)

Saved processed splits to /home/prasingh/data/AdvNLP/data/processed


## 5. Sanity Check — Load Back and Verify

In [9]:
from src.data_utils import load_jsonl_as_hf_dataset

train_ds = load_jsonl_as_hf_dataset(PROCESSED_DIR / 'train.jsonl')
val_ds   = load_jsonl_as_hf_dataset(PROCESSED_DIR / 'val.jsonl')
test_ds  = load_jsonl_as_hf_dataset(PROCESSED_DIR / 'test.jsonl')

print('Train dataset:', train_ds)
print('Val dataset:  ', val_ds)
print('Test dataset: ', test_ds)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train dataset: Dataset({
    features: ['messages'],
    num_rows: 40084
})
Val dataset:   Dataset({
    features: ['messages'],
    num_rows: 2234
})
Test dataset:  Dataset({
    features: ['messages'],
    num_rows: 3515
})
